In [0]:
use catalog ecommerce_dev;
create schema if not exists kpi

In [0]:
CREATE OR REPLACE VIEW ecommerce_dev.kpi.vw_deduped_orders AS
SELECT *
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_purchase_timestamp) AS rn
  FROM ecommerce_dev.gold.olist_datacube
)
WHERE rn = 1;

In [0]:
select * from ecommerce_dev.gold.olist_datacube;

## 1 | Total revenue by month

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.total_revenue_by_month AS
SELECT year(order_purchase_timestamp) as order_year, order_month AS month_number, SUM(total_payment_value) AS revenue_per_month
FROM ecommerce_dev.kpi.vw_deduped_orders
GROUP BY year(order_purchase_timestamp), order_month
ORDER BY order_year, order_month ASC;

select * from ecommerce_dev.kpi.total_revenue_by_month

## 2 | Average Order Value (AOV)

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.avg_order_value AS
WITH total_rev AS (
  SELECT SUM(total_payment_value) AS total FROM ecommerce_dev.kpi.vw_deduped_orders
),
no_of_orders AS (
  SELECT COUNT(DISTINCT order_id) AS total_orders FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT total_rev.total / no_of_orders.total_orders AS avg_order_value FROM total_rev, no_of_orders;

select * from ecommerce_dev.kpi.avg_order_value

## 3 | Top 10 products by revenue


In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.top10_products_by_revenue AS
SELECT product_id, product_category_name AS product_name, SUM(total_payment_value) AS total_revenue
FROM ecommerce_dev.kpi.vw_deduped_orders
GROUP BY product_id, product_category_name
ORDER BY SUM(total_payment_value) DESC
LIMIT 10;

select * from ecommerce_dev.kpi.top10_products_by_revenue

## 4 | Top 10 sellers by revenue

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.top10_sellers_by_revenue AS
SELECT seller_id, SUM(total_payment_value) AS total_revenue
FROM ecommerce_dev.kpi.vw_deduped_orders
GROUP BY seller_id
ORDER BY SUM(total_payment_value) DESC
LIMIT 10;

select * from ecommerce_dev.kpi.top10_sellers_by_revenue

## 5 | Orders by customer city and state

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.orders_by_customer_city_state AS
SELECT customer_state, customer_city, SUM(total_payment_value) AS total_revenue
FROM ecommerce_dev.kpi.vw_deduped_orders
GROUP BY customer_state, customer_city
ORDER BY customer_state, customer_city;

## 6 | New vs returning customers (monthly)

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.new_vs_returning_customers AS
WITH customer_first_purchase AS (
    SELECT customer_id, MIN(order_purchase_timestamp) AS first_purchase_date
    FROM ecommerce_dev.kpi.vw_deduped_orders
    GROUP BY customer_id
),
order_with_flag AS (
    SELECT o.order_id, o.customer_id,
           DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
           DATE_TRUNC('month', c.first_purchase_date) AS first_purchase_month
    FROM ecommerce_dev.kpi.vw_deduped_orders AS o
    JOIN customer_first_purchase AS c ON o.customer_id = c.customer_id
)
SELECT order_month,
       COUNT(DISTINCT CASE WHEN order_month = first_purchase_month THEN customer_id END) AS new_customers,
       COUNT(DISTINCT CASE WHEN order_month > first_purchase_month THEN customer_id END) AS existing_customers
FROM order_with_flag
GROUP BY order_month
ORDER BY order_month ASC;

select * from ecommerce_dev.kpi.new_vs_returning_customers

## 7 | Customer Lifetime Value (CLV)

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.customer_lifetime_value AS
WITH tr AS (
  SELECT SUM(total_payment_value) AS total_revenue,
         COUNT(DISTINCT order_id) AS total_orders,
         COUNT(DISTINCT customer_id) AS total_customers,
         DATEDIFF(year, MIN(order_purchase_timestamp), MAX(order_delivered_customer_date)) AS active_years
  FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT total_revenue / total_orders AS apv,
       total_orders / total_customers AS pf,
       active_years / total_customers AS cls,
       (total_revenue / total_orders) * (total_orders / total_customers) * (active_years / total_customers) AS customer_lifetime_value
FROM tr;

select * from ecommerce_dev.kpi.customer_lifetime_value;

## 8 | Order fulfillment rate


In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.order_fulfillment_rate AS
WITH delivered_count AS (
  SELECT COUNT(*) AS isdel_count FROM ecommerce_dev.kpi.vw_deduped_orders WHERE is_delivered = true
),
total_count AS (
  SELECT COUNT(*) AS total FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT delivered_count.isdel_count * 100 / total_count.total AS order_fulfillment_rate
FROM delivered_count, total_count;

select * from ecommerce_dev.kpi.order_fulfillment_rate;

## 9 | Late delivery percentage

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.late_delivery_percentage AS
WITH late_del_count AS (
  SELECT COUNT(*) AS isdel_count FROM ecommerce_dev.kpi.vw_deduped_orders WHERE is_late_delivery = false
),
total_count AS (
  SELECT COUNT(*) AS total FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT late_del_count.isdel_count * 100 / total_count.total AS late_delivery_percentage
FROM late_del_count, total_count;

select * from ecommerce_dev.kpi.late_delivery_percentage;

## 10 | Average delivery time (days)  --region**

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.avg_delivery_time_by_region AS
WITH delivery_days AS (
  SELECT *, DATEDIFF(day, order_purchase_timestamp, order_delivered_customer_date) AS del_day
  FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT customer_state, customer_city, AVG(del_day) AS avg_delivery_time_in_day
FROM delivery_days
GROUP BY customer_state, customer_city
ORDER BY customer_state, customer_city;

select * from ecommerce_dev.kpi.avg_delivery_time_by_region;

## 11 | Payment method distribution

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.payment_method_distribution AS
SELECT payment_type, COUNT(*) AS payment_count
FROM ecommerce_dev.kpi.vw_deduped_orders
WHERE payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY payment_type;

select * from ecommerce_dev.kpi.payment_method_distribution;

## 12 | Average review score by product category

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.avg_review_score_by_category AS
SELECT product_category_name, AVG(review_score) AS avg_review_score
FROM ecommerce_dev.kpi.vw_deduped_orders
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name;

select * from ecommerce_dev.kpi.avg_review_score_by_category;

## 13 | Average review score by seller

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.avg_review_score_by_seller AS
SELECT seller_id, AVG(review_score) AS avg_review_score
FROM ecommerce_dev.kpi.vw_deduped_orders
WHERE seller_id IS NOT NULL
GROUP BY seller_id
ORDER BY seller_id;

select * from ecommerce_dev.kpi.avg_review_score_by_seller;

## 14 | Percentage of low-rated orders (rating < 3)

In [0]:
CREATE OR REPLACE TABLE ecommerce_dev.kpi.low_rated_orders_percentage AS
WITH neg AS (
  SELECT COUNT(*) AS n_count FROM ecommerce_dev.kpi.vw_deduped_orders WHERE review_category = 'negative'
),
tot AS (
  SELECT COUNT(*) AS t_count FROM ecommerce_dev.kpi.vw_deduped_orders
)
SELECT neg.n_count * 100 / tot.t_count AS negative_review_percentage FROM neg, tot;

select * from ecommerce_dev.kpi.low_rated_orders_percentage;